In [0]:
from pyspark.sql import functions as F

sales_df = spark.table("bronze_sales")

channels_df = spark.table("bronze_channels")

display(sales_df.limit(5))
display(channels_df.limit(5))

��D A T E , C E _ B R A N D _ F L V R , B R A N D _ N M , B t l r _ O r g _ L V L _ C _ D e s c , C H N L _ G R O U P , T R A D E _ C H N L _ D E S C , P K G _ C A T , P k g _ C a t _ D e s c , T S R _ P C K G _ N M , $   V o l u m e , Y E A R , M O N T H , P E R I O D 
 0 1 / 0 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , C A N A D A , L E I S U R E , S P O R T   V E N U E , N 2 0 O , 2 0 Z / 6 0 0 M L , . 5 9 1 L   N R P   2 4 L , 2 2 . 4 8 , 2 0 0 6 ,1.0,1.0
 0 1 / 0 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , N O R T H E A S T , S U P E R S , S U P E R E T T E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 1 0 0 , 2 0 0 6 ,1.0,1.0
 0 1 / 0 1 / 2 0 0 6 , 3 5 5 4 ,   S T R A W B E R R Y , S O U T H E A S T , W O R K P L A C E , P L A N T   /   O F F I C E , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 6 6 . 1 4 , 2 0 0 6 ,1.0,1.0
 0 1 / 0 1 / 2 0 0 6 , 3 4 4 1 ,   R A S P B E R R Y , M I D W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 2 2 2 . 5 , 2 0 0 6 ,1.0,1.0
 0 1 / 0 1 / 2 0 0 6 , 3 4 4 0 ,   L E M O N , W E S T , M A S S   M E R C H A N D I S E R , M A S S   M E R C H A N D I S E R , N 2 0 O , 2 0 Z / 6 0 0 M L , 2 0 Z   N R P   2 4 L , 3 0 2 . 5 , 2 0 0 6 ,1.0,1.0


TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,SERVICES,MIX
PLANT / OFFICE,SERVICES,MIX
MASS MERCHANDISER,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC


In [0]:
channels_df.columns

['TRADE_CHNL_DESC', 'TRADE_GROUP_DESC', 'TRADE_TYPE_DESC']

In [0]:
import re

sales_df_clean = sales_df.toDF(
    *[re.sub(r'\x00', '', col).replace('��', '') for col in sales_df.columns]
)

sales_df_clean.columns

['DATE',
 'CE_BRAND_FLVR',
 'BRAND_NM',
 'Btlr_Org_LVL_C_Desc',
 'CHNL_GROUP',
 'TRADE_CHNL_DESC',
 'PKG_CAT',
 'Pkg_Cat_Desc',
 'TSR_PCKG_NM',
 '$ Volume',
 'YEAR',
 'MONTH',
 'PERIOD']

In [0]:
(
    sales_df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_sales")
)

In [0]:
sales_df = spark.table("bronze_sales")

sales_df.columns

['DATE',
 'CE_BRAND_FLVR',
 'BRAND_NM',
 'Btlr_Org_LVL_C_Desc',
 'CHNL_GROUP',
 'TRADE_CHNL_DESC',
 'PKG_CAT',
 'Pkg_Cat_Desc',
 'TSR_PCKG_NM',
 '$ Volume',
 'YEAR',
 'MONTH',
 'PERIOD']

In [0]:
sales_df.select("TRADE_CHNL_DESC").distinct().count()

31

In [0]:
silver_df = (
    sales_df
    .join(
        channels_df,
        "TRADE_CHNL_DESC",
        "left"
    )
)

In [0]:
display(silver_df.limit(10))

TRADE_CHNL_DESC,DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,$ Volume,YEAR,MONTH,PERIOD,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,2006-01-01,3440,LEMON,CANADA,LEISURE,N20O,20Z/600ML,.591L NRP 24L,22.48,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,2006-01-01,3440,LEMON,NORTHEAST,SUPERS,N20O,20Z/600ML,20Z NRP 24L,100.0,2006,1,1,SERVICES,MIX
PLANT / OFFICE,2006-01-01,3554,STRAWBERRY,SOUTHEAST,WORKPLACE,N20O,20Z/600ML,20Z NRP 24L,66.14,2006,1,1,SERVICES,MIX
MASS MERCHANDISER,2006-01-01,3441,RASPBERRY,MIDWEST,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,222.5,2006,1,1,GROCERY,MIX
MASS MERCHANDISER,2006-01-01,3440,LEMON,WEST,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,302.5,2006,1,1,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,2006-01-01,3440,LEMON,MIDWEST,OTHER SMALL STORES,N20O,20Z/600ML,20Z NRP 24L,10.0,2006,1,1,GROCERY,ALCOHOLIC
CONVENIENCE STORE,2006-01-01,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,SERVICES,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,CANADA,FOOD SERVICE,N20O,20Z/600ML,.591L NRP 24L,59.95,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,2006-01-01,3554,STRAWBERRY,SOUTHWEST,SUPERS,N20O,20Z/600ML,20z NRP 24L S,300.0,2006,1,1,GROCERY,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,SOUTHEAST,FOOD SERVICE,N20O,20Z/600ML,20Z NRP 24L,85.0,2006,1,1,ENTERTAINMENT,ALCOHOLIC


In [0]:
print("Bronze:", sales_df.count())
print("Silver:", silver_df.count())

Bronze: 16151
Silver: 16151


In [0]:
dim_brand = (
    silver_df
    .select("BRAND_NM")
    .distinct()
)

In [0]:
dim_region = (
    silver_df
    .select(
        "Btlr_Org_LVL_C_Desc"
    )
    .distinct()
)

In [0]:
dim_channel = (
    silver_df
    .select(
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "TRADE_TYPE_DESC"
    )
    .distinct()
)

In [0]:
fact_sales = (
    silver_df
    .groupBy(
        "DATE",
        "BRAND_NM",
        "Btlr_Org_LVL_C_Desc",
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC"
    )
    .agg(
        F.sum("$ Volume")
        .alias("SALES_VOLUME")
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy(
    "Btlr_Org_LVL_C_Desc"
).orderBy(
    F.desc("SALES_VOLUME")
)

top3 = (
    fact_sales
    .groupBy(
        "Btlr_Org_LVL_C_Desc",
        "TRADE_GROUP_DESC"
    )
    .agg(
        F.sum("SALES_VOLUME")
        .alias("TOTAL_VOLUME")
    )
    .withColumn(
        "rank",
        F.row_number().over(window)
    )
    .filter("rank <= 3")
)

Gold

In [0]:
dim_brand = (
    silver_df
    .select("BRAND_NM")
    .distinct()
)

display(dim_brand)

BRAND_NM
LEMON
STRAWBERRY
RASPBERRY
GRAPE


In [0]:
(
    dim_brand
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_brand")
)

In [0]:
spark.table("dim_brand").count()

4

In [0]:
dim_region = (
    silver_df
    .select(
        F.col("Btlr_Org_LVL_C_Desc")
        .alias("REGION")
    )
    .distinct()
)

display(dim_region)

REGION
CANADA
NORTHEAST
SOUTHEAST
MIDWEST
WEST
SOUTHWEST
GREAT LAKES


In [0]:
(
    dim_region
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_region")
)

In [0]:
dim_channel = (
    silver_df
    .select(
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "TRADE_TYPE_DESC"
    )
    .distinct()
)

display(dim_channel)

TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC
CLUB STORE,ENTERTAINMENT,ALCOHOLIC
PLANT / OFFICE,SERVICES,MIX
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC
MILITARY-OTHER,GOV & MILITARY,MIX
RETAIL SPECIALITY SVCS,GROCERY,ALCOHOLIC
SUPERMARKET,GROCERY,MIX
ALL OTHER,OTHER,MIX
TRANSPORTATION,SERVICES,NON ALCOHOLIC
COLLEGE/UNIVERSITY,ACADEMIC,NON ALCOHOLIC
LOCAL+TRADITIONAL GROCERY,GROCERY,MIX


In [0]:
(
    dim_channel
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_channel")
)

In [0]:
display(dim_brand)
display(dim_region)
display(dim_channel)

BRAND_NM
LEMON
STRAWBERRY
RASPBERRY
GRAPE


REGION
CANADA
NORTHEAST
SOUTHEAST
MIDWEST
WEST
SOUTHWEST
GREAT LAKES


TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC
CLUB STORE,ENTERTAINMENT,ALCOHOLIC
PLANT / OFFICE,SERVICES,MIX
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC
MILITARY-OTHER,GOV & MILITARY,MIX
RETAIL SPECIALITY SVCS,GROCERY,ALCOHOLIC
SUPERMARKET,GROCERY,MIX
ALL OTHER,OTHER,MIX
TRANSPORTATION,SERVICES,NON ALCOHOLIC
COLLEGE/UNIVERSITY,ACADEMIC,NON ALCOHOLIC
LOCAL+TRADITIONAL GROCERY,GROCERY,MIX


In [0]:
from pyspark.sql import functions as F

# Dim Brand
dim_brand = (
    silver_df
    .select("BRAND_NM")
    .distinct()
)

(
    dim_brand
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_brand")
)


# Dim Region
dim_region = (
    silver_df
    .select(
        F.col("Btlr_Org_LVL_C_Desc").alias("REGION")
    )
    .distinct()
)

(
    dim_region
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_region")
)


# Dim Channel
dim_channel = (
    silver_df
    .select(
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "TRADE_TYPE_DESC"
    )
    .distinct()
)

(
    dim_channel
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_channel")
)

In [0]:
fact_sales = (
    silver_df
    .groupBy(
        "DATE",
        "BRAND_NM",
        "Btlr_Org_LVL_C_Desc",
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "YEAR",
        "MONTH"
    )
    .agg(
        F.sum("$ Volume").alias("SALES_VOLUME")
    )
)

display(fact_sales.limit(10))

DATE,BRAND_NM,Btlr_Org_LVL_C_Desc,TRADE_CHNL_DESC,TRADE_GROUP_DESC,YEAR,MONTH,SALES_VOLUME
2006-01-01,STRAWBERRY,SOUTHWEST,PRIMARY/SECONDARY SCHOOL,ACADEMIC,2006,1,735.18
2006-01-01,LEMON,NORTHEAST,SUPERMARKET,GROCERY,2006,1,5309.91
2006-01-01,LEMON,NORTHEAST,LIQUOR/BEER/WINE/SOFT,GROCERY,2006,1,56.34
2006-01-01,LEMON,CANADA,MASS MERCHANDISER,GROCERY,2006,1,573.94
2006-01-01,LEMON,GREAT LAKES,SUPERMARKET,GROCERY,2006,1,4785.17
2006-01-01,LEMON,NORTHEAST,LOCAL+TRADITIONAL GROCERY,GROCERY,2006,1,605.0
2006-01-01,LEMON,MIDWEST,MASS MERCHANDISER,GROCERY,2006,1,553.15
2006-01-01,STRAWBERRY,MIDWEST,GOVERNMENT(NON-MILITARY),GOV & MILITARY,2006,1,5.0
2006-01-01,LEMON,SOUTHWEST,MILITARY-COMMISARY,GOV & MILITARY,2006,1,68.6
2006-01-01,LEMON,WEST,SUPERMARKET,GROCERY,2006,1,1704.77


In [0]:
fact_sales.count()

7912

In [0]:
silver_df.count()

16151

In [0]:
(
    fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_beverage_sales")
)

In [0]:
fact_sales = (
    silver_df
    .groupBy(
        "DATE",
        "BRAND_NM",
        "Btlr_Org_LVL_C_Desc",
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "YEAR",
        "MONTH"
    )
    .agg(
        F.sum("$ Volume").alias("SALES_VOLUME")
    )
)

display(fact_sales.limit(10))

DATE,BRAND_NM,Btlr_Org_LVL_C_Desc,TRADE_CHNL_DESC,TRADE_GROUP_DESC,YEAR,MONTH,SALES_VOLUME
2006-01-01,STRAWBERRY,SOUTHWEST,PRIMARY/SECONDARY SCHOOL,ACADEMIC,2006,1,735.18
2006-01-01,LEMON,NORTHEAST,SUPERMARKET,GROCERY,2006,1,5309.91
2006-01-01,LEMON,NORTHEAST,LIQUOR/BEER/WINE/SOFT,GROCERY,2006,1,56.34
2006-01-01,LEMON,CANADA,MASS MERCHANDISER,GROCERY,2006,1,573.94
2006-01-01,LEMON,GREAT LAKES,SUPERMARKET,GROCERY,2006,1,4785.17
2006-01-01,LEMON,NORTHEAST,LOCAL+TRADITIONAL GROCERY,GROCERY,2006,1,605.0
2006-01-01,LEMON,MIDWEST,MASS MERCHANDISER,GROCERY,2006,1,553.15
2006-01-01,STRAWBERRY,MIDWEST,GOVERNMENT(NON-MILITARY),GOV & MILITARY,2006,1,5.0
2006-01-01,LEMON,SOUTHWEST,MILITARY-COMMISARY,GOV & MILITARY,2006,1,68.6
2006-01-01,LEMON,WEST,SUPERMARKET,GROCERY,2006,1,1704.77


In [0]:
fact_sales.count()

7912

In [0]:
silver_df.count()

16151

In [0]:
(
    fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_beverage_sales")
)

In [0]:
display(fact_sales.limit(10))

DATE,BRAND_NM,Btlr_Org_LVL_C_Desc,TRADE_CHNL_DESC,TRADE_GROUP_DESC,YEAR,MONTH,SALES_VOLUME
2006-01-01,STRAWBERRY,SOUTHWEST,PRIMARY/SECONDARY SCHOOL,ACADEMIC,2006,1,735.18
2006-01-01,LEMON,NORTHEAST,SUPERMARKET,GROCERY,2006,1,5309.91
2006-01-01,LEMON,NORTHEAST,LIQUOR/BEER/WINE/SOFT,GROCERY,2006,1,56.34
2006-01-01,LEMON,CANADA,MASS MERCHANDISER,GROCERY,2006,1,573.94
2006-01-01,LEMON,GREAT LAKES,SUPERMARKET,GROCERY,2006,1,4785.17
2006-01-01,LEMON,NORTHEAST,LOCAL+TRADITIONAL GROCERY,GROCERY,2006,1,605.0
2006-01-01,LEMON,MIDWEST,MASS MERCHANDISER,GROCERY,2006,1,553.15
2006-01-01,STRAWBERRY,MIDWEST,GOVERNMENT(NON-MILITARY),GOV & MILITARY,2006,1,5.0
2006-01-01,LEMON,SOUTHWEST,MILITARY-COMMISARY,GOV & MILITARY,2006,1,68.6
2006-01-01,LEMON,WEST,SUPERMARKET,GROCERY,2006,1,1704.77


In [0]:
agg_brand_month_sales = (
    fact_sales
    .groupBy(
        "YEAR",
        "MONTH",
        "BRAND_NM"
    )
    .agg(
        F.sum("SALES_VOLUME")
        .alias("TOTAL_SALES_VOLUME")
    )
)

display(agg_brand_month_sales.limit(20))

YEAR,MONTH,BRAND_NM,TOTAL_SALES_VOLUME
2006,1,LEMON,505845.2199999979
2006,1,RASPBERRY,392551.69999999914
2006,2,LEMON,550940.8499999979
2006,2,STRAWBERRY,345518.39999999915
2006,3,STRAWBERRY,475032.17999999865
2006,1,STRAWBERRY,315902.64999999927
2006,2,RASPBERRY,409661.5199999984
2006,3,GRAPE,7.5
2006,3,RASPBERRY,587522.5599999976
2006,3,LEMON,765942.5799999965


In [0]:
(
    agg_brand_month_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("agg_brand_month_sales")
)

In [0]:
agg_region_trade_sales = (
    fact_sales
    .groupBy(
        "Btlr_Org_LVL_C_Desc",
        "TRADE_GROUP_DESC"
    )
    .agg(
        F.sum("SALES_VOLUME")
        .alias("TOTAL_SALES_VOLUME")
    )
)

display(agg_region_trade_sales.limit(20))

Btlr_Org_LVL_C_Desc,TRADE_GROUP_DESC,TOTAL_SALES_VOLUME
CANADA,ENTERTAINMENT,38670.21000000005
WEST,ENTERTAINMENT,47602.42000000003
GREAT LAKES,GROCERY,380297.1899999994
WEST,OTHER,25087.160000000003
WEST,SERVICES,87972.77000000006
CANADA,GOV & MILITARY,1247.0600000000015
SOUTHEAST,OTHER,3919.729999999999
SOUTHWEST,GROCERY,271246.6799999998
SOUTHWEST,GOV & MILITARY,9901.220000000003
CANADA,ACADEMIC,53117.56000000003


In [0]:
(
    agg_region_trade_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("agg_region_trade_sales")
)

In [0]:
display(agg_brand_month_sales.limit(10))

YEAR,MONTH,BRAND_NM,TOTAL_SALES_VOLUME
2006,1,LEMON,505845.2199999979
2006,1,RASPBERRY,392551.69999999914
2006,2,LEMON,550940.8499999979
2006,2,STRAWBERRY,345518.39999999915
2006,3,STRAWBERRY,475032.17999999865
2006,1,STRAWBERRY,315902.64999999927
2006,2,RASPBERRY,409661.5199999984
2006,3,GRAPE,7.5
2006,3,RASPBERRY,587522.5599999976
2006,3,LEMON,765942.5799999965


In [0]:
display(agg_region_trade_sales.limit(10))

Btlr_Org_LVL_C_Desc,TRADE_GROUP_DESC,TOTAL_SALES_VOLUME
CANADA,ENTERTAINMENT,38670.21000000005
WEST,ENTERTAINMENT,47602.42000000003
GREAT LAKES,GROCERY,380297.1899999994
WEST,OTHER,25087.160000000003
WEST,SERVICES,87972.77000000006
CANADA,GOV & MILITARY,1247.0600000000015
SOUTHEAST,OTHER,3919.729999999999
SOUTHWEST,GROCERY,271246.6799999998
SOUTHWEST,GOV & MILITARY,9901.220000000003
CANADA,ACADEMIC,53117.56000000003


In [0]:
from pyspark.sql.window import Window

window_region = Window \
    .partitionBy("Btlr_Org_LVL_C_Desc") \
    .orderBy(F.desc("TOTAL_SALES_VOLUME"))


top3_trade_groups = (
    agg_region_trade_sales
    .withColumn(
        "RANK",
        F.row_number().over(window_region)
    )
    .filter(
        F.col("RANK") <= 3
    )
)

display(top3_trade_groups)

Btlr_Org_LVL_C_Desc,TRADE_GROUP_DESC,TOTAL_SALES_VOLUME,RANK
CANADA,GROCERY,168640.07000000007,1
CANADA,SERVICES,83748.06999999998,2
CANADA,ACADEMIC,53117.56000000003,3
GREAT LAKES,GROCERY,380297.1899999994,1
GREAT LAKES,SERVICES,157798.63999999984,2
GREAT LAKES,ACADEMIC,152043.01000000004,3
MIDWEST,GROCERY,326351.38999999943,1
MIDWEST,SERVICES,129258.59999999999,2
MIDWEST,ACADEMIC,88605.36999999992,3
NORTHEAST,GROCERY,403710.6499999998,1


In [0]:
display(
    agg_brand_month_sales
    .orderBy(
        "YEAR",
        "MONTH",
        "BRAND_NM"
    )
)

YEAR,MONTH,BRAND_NM,TOTAL_SALES_VOLUME
2006,1,LEMON,505845.2199999979
2006,1,RASPBERRY,392551.69999999914
2006,1,STRAWBERRY,315902.64999999927
2006,2,LEMON,550940.8499999979
2006,2,RASPBERRY,409661.5199999984
2006,2,STRAWBERRY,345518.39999999915
2006,3,GRAPE,7.5
2006,3,LEMON,765942.5799999965
2006,3,RASPBERRY,587522.5599999976
2006,3,STRAWBERRY,475032.17999999865


In [0]:
region_brand_sales = (
    fact_sales
    .groupBy(
        "Btlr_Org_LVL_C_Desc",
        "BRAND_NM"
    )
    .agg(
        F.sum("SALES_VOLUME")
        .alias("TOTAL_SALES_VOLUME")
    )
)

In [0]:
window_lowest = Window \
    .partitionBy("Btlr_Org_LVL_C_Desc") \
    .orderBy(F.asc("TOTAL_SALES_VOLUME"))


lowest_brand_region = (
    region_brand_sales
    .withColumn(
        "RANK",
        F.row_number().over(window_lowest)
    )
    .filter(
        F.col("RANK") == 1
    )
)

display(lowest_brand_region)

Btlr_Org_LVL_C_Desc,BRAND_NM,TOTAL_SALES_VOLUME,RANK
CANADA,STRAWBERRY,74009.61,1
GREAT LAKES,STRAWBERRY,208747.89999999985,1
MIDWEST,STRAWBERRY,149206.7799999999,1
NORTHEAST,STRAWBERRY,195581.48999999985,1
SOUTHEAST,RASPBERRY,223958.50000000003,1
SOUTHWEST,GRAPE,7.5,1
WEST,STRAWBERRY,134511.17999999988,1


In [0]:
fact_sales = spark.table("fact_beverage_sales")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

CREATE SCHEMA IF NOT EXISTS silver;

CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
SHOW SCHEMAS;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
%sql
SHOW DATABASES;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
%sql
ALTER TABLE bronze.dim_brand RENAME TO gold.dim_brand;





In [0]:
sales_df = spark.table("bronze.beverage_sales")

channels_df = spark.table("bronze.beverage_channels")

silver_df = (
    sales_df
    .join(
        channels_df,
        "TRADE_CHNL_DESC",
        "left"
    )
)

In [0]:
from pyspark.sql import functions as F

sales_df = spark.table("bronze.beverage_sales")
channels_df = spark.table("bronze.beverage_channels")

In [0]:
silver_df = (
    sales_df
    .join(
        channels_df,
        "TRADE_CHNL_DESC",
        "left"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.beverage_sales")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5919868483873306>, line 6
      1 (
      2     silver_df
      3     .write
      4     .format("delta")
      5     .mode("overwrite")
----> 6     .saveAsTable("silver.beverage_sales")
      7 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     r

In [0]:
sales_df = spark.table("bronze.beverage_sales")
channels_df = spark.table("bronze.beverage_channels")

In [0]:
sales_df = spark.table("default.bronze_sales")

channels_df = spark.table("default.bronze_channels")

In [0]:
silver_df = (
    sales_df
    .join(
        channels_df,
        "TRADE_CHNL_DESC",
        "left"
    )
)

In [0]:
sales_df = spark.table("bronze.beverage_sales")

channels_df = spark.table("bronze.beverage_channels")

In [0]:
sales_df = spark.table("bronze.bronze_sales")

channels_df = spark.table("bronze.bronze_channels")

In [0]:
silver_df = (
    sales_df
    .join(
        channels_df,
        "TRADE_CHNL_DESC",
        "left"
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumnRenamed("$ Volume", "SALES_VOLUME")
)

In [0]:
silver_df.columns

['TRADE_CHNL_DESC',
 'DATE',
 'CE_BRAND_FLVR',
 'BRAND_NM',
 'Btlr_Org_LVL_C_Desc',
 'CHNL_GROUP',
 'PKG_CAT',
 'Pkg_Cat_Desc',
 'TSR_PCKG_NM',
 'SALES_VOLUME',
 'YEAR',
 'MONTH',
 'PERIOD',
 'TRADE_GROUP_DESC',
 'TRADE_TYPE_DESC']

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.beverage_sales")
)

In [0]:
fact_sales = (
    silver_df
    .groupBy(
        "DATE",
        "BRAND_NM",
        "Btlr_Org_LVL_C_Desc",
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "YEAR",
        "MONTH"
    )
    .agg(
        F.sum("SALES_VOLUME").alias("SALES_VOLUME")
    )
)

display(fact_sales.limit(10))

DATE,BRAND_NM,Btlr_Org_LVL_C_Desc,TRADE_CHNL_DESC,TRADE_GROUP_DESC,YEAR,MONTH,SALES_VOLUME
2006-01-01,STRAWBERRY,SOUTHWEST,PRIMARY/SECONDARY SCHOOL,ACADEMIC,2006,1,735.18
2006-01-01,LEMON,NORTHEAST,SUPERMARKET,GROCERY,2006,1,5309.91
2006-01-01,LEMON,NORTHEAST,LIQUOR/BEER/WINE/SOFT,GROCERY,2006,1,56.34
2006-01-01,LEMON,CANADA,MASS MERCHANDISER,GROCERY,2006,1,573.94
2006-01-01,LEMON,GREAT LAKES,SUPERMARKET,GROCERY,2006,1,4785.17
2006-01-01,LEMON,NORTHEAST,LOCAL+TRADITIONAL GROCERY,GROCERY,2006,1,605.0
2006-01-01,LEMON,MIDWEST,MASS MERCHANDISER,GROCERY,2006,1,553.15
2006-01-01,STRAWBERRY,MIDWEST,GOVERNMENT(NON-MILITARY),GOV & MILITARY,2006,1,5.0
2006-01-01,LEMON,SOUTHWEST,MILITARY-COMMISARY,GOV & MILITARY,2006,1,68.6
2006-01-01,LEMON,WEST,SUPERMARKET,GROCERY,2006,1,1704.77


In [0]:
(
    fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.fact_beverage_sales")
)

In [0]:
agg_brand_month_sales = (
    fact_sales
    .groupBy(
        "YEAR",
        "MONTH",
        "BRAND_NM"
    )
    .agg(
        F.sum("SALES_VOLUME").alias("TOTAL_SALES_VOLUME")
    )
)

(
    agg_brand_month_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.agg_brand_month_sales")
)

In [0]:
agg_region_trade_sales = (
    fact_sales
    .groupBy(
        "Btlr_Org_LVL_C_Desc",
        "TRADE_GROUP_DESC"
    )
    .agg(
        F.sum("SALES_VOLUME").alias("TOTAL_SALES_VOLUME")
    )
)

(
    agg_region_trade_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.agg_region_trade_sales")
)

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.beverage_sales")

display(silver_df.limit(10))

TRADE_CHNL_DESC,DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,SALES_VOLUME,YEAR,MONTH,PERIOD,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,2006-01-01,3440,LEMON,CANADA,LEISURE,N20O,20Z/600ML,.591L NRP 24L,22.48,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,2006-01-01,3440,LEMON,NORTHEAST,SUPERS,N20O,20Z/600ML,20Z NRP 24L,100.0,2006,1,1,SERVICES,MIX
PLANT / OFFICE,2006-01-01,3554,STRAWBERRY,SOUTHEAST,WORKPLACE,N20O,20Z/600ML,20Z NRP 24L,66.14,2006,1,1,SERVICES,MIX
MASS MERCHANDISER,2006-01-01,3441,RASPBERRY,MIDWEST,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,222.5,2006,1,1,GROCERY,MIX
MASS MERCHANDISER,2006-01-01,3440,LEMON,WEST,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,302.5,2006,1,1,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,2006-01-01,3440,LEMON,MIDWEST,OTHER SMALL STORES,N20O,20Z/600ML,20Z NRP 24L,10.0,2006,1,1,GROCERY,ALCOHOLIC
CONVENIENCE STORE,2006-01-01,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,SERVICES,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,CANADA,FOOD SERVICE,N20O,20Z/600ML,.591L NRP 24L,59.95,2006,1,1,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,2006-01-01,3554,STRAWBERRY,SOUTHWEST,SUPERS,N20O,20Z/600ML,20z NRP 24L S,300.0,2006,1,1,GROCERY,MIX
QUICK SERVICE RESTAURANT,2006-01-01,3554,STRAWBERRY,SOUTHEAST,FOOD SERVICE,N20O,20Z/600ML,20Z NRP 24L,85.0,2006,1,1,ENTERTAINMENT,ALCOHOLIC


In [0]:
fact_sales = (
    silver_df
    .groupBy(
        "DATE",
        "BRAND_NM",
        "Btlr_Org_LVL_C_Desc",
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "YEAR",
        "MONTH"
    )
    .agg(
        F.sum("SALES_VOLUME").alias("SALES_VOLUME")
    )
)

display(fact_sales.limit(10))

DATE,BRAND_NM,Btlr_Org_LVL_C_Desc,TRADE_CHNL_DESC,TRADE_GROUP_DESC,YEAR,MONTH,SALES_VOLUME
2006-01-01,LEMON,CANADA,SPORT VENUE,ENTERTAINMENT,2006,1,29.97
2006-01-01,LEMON,NORTHEAST,SUPERETTE,SERVICES,2006,1,565.72
2006-01-01,STRAWBERRY,SOUTHEAST,PLANT / OFFICE,SERVICES,2006,1,223.04000000000002
2006-01-01,RASPBERRY,MIDWEST,MASS MERCHANDISER,GROCERY,2006,1,539.51
2006-01-01,LEMON,WEST,MASS MERCHANDISER,GROCERY,2006,1,558.22
2006-01-01,LEMON,MIDWEST,LIQUOR/BEER/WINE/SOFT,GROCERY,2006,1,10.0
2006-01-01,RASPBERRY,SOUTHEAST,CONVENIENCE STORE,SERVICES,2006,1,2805.7799999999997
2006-01-01,STRAWBERRY,CANADA,QUICK SERVICE RESTAURANT,ENTERTAINMENT,2006,1,99.91
2006-01-01,STRAWBERRY,SOUTHWEST,SUPERMARKET,GROCERY,2006,1,1465.71
2006-01-01,STRAWBERRY,SOUTHEAST,QUICK SERVICE RESTAURANT,ENTERTAINMENT,2006,1,245.0


In [0]:
(
    fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.fact_beverage_sales")
)

In [0]:
%sql
SELECT COUNT (*)
FROM gold.fact_beverage_sales;


COUNT(*)
7912


In [0]:
dim_channel = (
    silver_df
    .select(
        "TRADE_CHNL_DESC",
        "TRADE_GROUP_DESC",
        "TRADE_TYPE_DESC"
    )
    .distinct()
)

display(dim_channel.limit(10))


TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,SERVICES,MIX
PLANT / OFFICE,SERVICES,MIX
MASS MERCHANDISER,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC
CONVENIENCE STORE,SERVICES,MIX
QUICK SERVICE RESTAURANT,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,GROCERY,MIX
ALL OTHER,OTHER,MIX
OTHER EATING + DRINKING,SERVICES,MIX


In [0]:
(
    dim_channel
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_channel")
)

In [0]:
%sql
SHOW TABLES IN gold;

database,tableName,isTemporary
gold,agg_brand_month_sales,false
gold,agg_region_trade_sales,false
gold,dim_brand,false
gold,dim_channel,false
gold,dim_region,false
gold,fact_beverage_sales,false
